# HST SN1983V Photometry Notebook

Sophia Kressy

Adapted JWST/MIRI photometry notebook from Justin Pierel to do HST photometry for SN 1983V

In [1]:
%matplotlib inline

* Create directory to store data -- make sure this matches your current working directory
* Import WEBBPSF Data

In [2]:
import sys,os,glob,shutil
import tarfile, urllib.request

# Set environmental variables
#os.environ["WEBBPSF_PATH"] = "./webbpsf-data/webbpsf-data"
#os.environ["PYSYN_CDBS"] = "./grp/redcat/trds/"

#os.environ["STPSF_PATH"] = "/grp/stpsf/stpsf-data"
# os.environ["STPSF_PATH"] = "./stpsf-data/stpsf-data/"
# os.environ["PYSYN_CDBS"] = "/grp/hst/cdbs"

# update this path to your working directory!!
os.environ["STPSF_PATH"] = '/Users/sophiakressy/Library/Mobile Documents/com~apple~CloudDocs/Desktop/VS_Workspace/sn_dust/stpsf-data/stpsf-data/'
os.environ["PYSYN_CDBS"] = "/Users/sophiakressy/Library/Mobile Documents/com~apple~CloudDocs/Desktop/VS_Workspace/sn_dust/grp"

# WEBBPSF Data
#boxlink = 'https://stsci.box.com/shared/static/qxpiaxsjwo15ml6m4pkhtk36c9jgj70k.gz'  
boxlink = 'https://stsci.box.com/shared/static/kqfolg2bfzqc4mjkgmujo06d3iaymahv.gz'
#boxfile = './webbpsf-data/webbpsf-data-1.0.0.tar.gz'
boxfile = './stpsf-data/stpsf-data-LATEST.tar.gz'
synphot_url = 'http://ssb.stsci.edu/trds/tarfiles/synphot5.tar.gz'
synphot_file = './synphot5.tar.gz'

#webbpsf_folder = './webbpsf-data'
stpsf_folder = "./stpsf-data"
synphot_folder = './grp'

# Gather webbpsf files
#psfExist = os.path.exists(webbpsf_folder)
psfExist = os.path.exists(stpsf_folder)
if not psfExist:
    os.makedirs(stpsf_folder)
    urllib.request.urlretrieve(boxlink, boxfile)
    gzf = tarfile.open(boxfile)
    gzf.extractall(stpsf_folder)

# Gather synphot files
synExist = os.path.exists(synphot_folder)
if not synExist:
    os.makedirs(synphot_folder)
    urllib.request.urlretrieve(synphot_url, synphot_file)
    gzf = tarfile.open(synphot_file)
    gzf.extractall('./')

Import functions

In [3]:
import numpy as np
import sys,os,glob
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.table import Table
from astropy.nddata import extract_array
from astropy.coordinates import SkyCoord
from astropy import wcs
from astropy.wcs.utils import skycoord_to_pixel
from astropy import units as u
# from astroquery.mast import Observations
from astropy.visualization import (simple_norm,LinearStretch)
from astropy.wcs import WCS
from photutils.centroids import centroid_com, centroid_sources, centroid_2dg, centroid_quadratic
from photutils.centroids import (centroid_1dg, centroid_2dg,
                                 centroid_com, centroid_quadratic)
from photutils.datasets import make_4gaussians_image
# import jhat
# from jhat import hst_photclass,st_wcs_align
import space_phot
from stwcs.updatewcs import wcsutil
import shutil
import drizzlepac
import stpsf


/Users/sophiakressy/miniconda3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm




The following task in the stsci.skypac package can be run with TEAL:
                                    skymatch                                    


**WARNING**: LOCAL JWST PRD VERSION PRDOPSSOC-068 DOESN'T MATCH THE CURRENT ONLINE VERSION PRDOPSSOC-071
Please consider updating pysiaf, e.g. pip install --upgrade pysiaf or conda update pysiaf


The following tasks in the drizzlepac package can be run with TEAL:
    astrodrizzle       config_testbed      imagefindpars           mapreg       
       photeq            pixreplace           pixtopix            pixtosky      
  refimagefindpars       resetbits          runastrodriz          skytopix      
     tweakback            tweakreg           updatenpol


* polyfit2d and polyval2D_custom: create polyfit functions for a 2D array (credit/questions: Justin Pierel)
* optimize_psf: takes Justin's code and loops through xshift, yshift, background parameters to find best values.
    - For each iteration, fits PSF and calculates residual. 
    - The residual is fitted with a 2D surface polynomial and calculates the residual.
    - The standard deviation of this residual of the residual is found; this is the value that is minimized.
    - options for plotting and number of iterations for each parameter.
    - window is assumed to be 11 pixels. This needs to be incorportated at some point as an additional parameter.

In [4]:
def polyfit2d(x, y, z, deg=2):
    # Build design matrix for 2D polynomial of given degree
    ncols = (deg + 1) * (deg + 2) // 2
    G = np.zeros((x.size, ncols))
    ij = [(i, j) for i in range(deg+1) for j in range(deg+1-i)]
    for k, (i, j) in enumerate(ij):
        G[:, k] = x**i * y**j
    m, _, _, _ = np.linalg.lstsq(G, z, rcond=None)
    return m, ij

def polyval2d_custom(x, y, m, ij):
    z = np.zeros_like(x, dtype=float)

    for a, (i, j) in zip(m, ij):
        z += a * x**i * y**j
    return z


def optimize_psf(drz_obs, drz_psf, sn_loc, fwidth, xshift_range, yshift_range, background_range, nxs, nys, nbkg, plotting):
    smooth_stds = []
    xshifts     = []
    yshifts     = []
    backgrounds = []

    for xshift in np.linspace(xshift_range[0],xshift_range[1],nxs):
        for yshift in np.linspace(yshift_range[0],yshift_range[1],nys): 
            for bkg in np.linspace(background_range[0],background_range[1],nbkg):
                drz_obs.psf_photometry(drz_psf,sn_loc,bounds={'flux':[-30000,40000],
                            'centroid':[-1,1],                              
                            'bkg':[-10,10]},
                            fit_width=fwidth,
                            background = bkg,
                            fit_bkg=False,
                            fit_centroid=False, 
                            fit_flux=True,
                            centroidx_shift=xshift,
                            centroidy_shift=yshift,
                            psf_method='nest',
                            center_weight=25,npoints=200)

                resid_cutout = drz_obs.psf_result.resid_arr[0][2:fwidth-2,2:fwidth-2]
                yy, xx = np.mgrid[0:resid_cutout.shape[0], 0:resid_cutout.shape[1]]
                xx = xx - resid_cutout.shape[1]//2
                yy = yy - resid_cutout.shape[0]//2

                # Fit and evaluate
                z = resid_cutout.flatten()
                x_flat = xx.flatten()
                y_flat = yy.flatten()
                m, ij = polyfit2d(x_flat, y_flat, z, deg=2)
                model = polyval2d_custom(xx, yy, m, ij)
                residuals_from_model = resid_cutout - model

                # Final metric: deviation from smooth model
                smoothness_metric = np.std(residuals_from_model)
                # print(f"Deviation from 2D smooth background model: {smoothness_metric:.5f}")
                
                smooth_stds.append(smoothness_metric)
                xshifts.append(xshift)
                yshifts.append(yshift)
                backgrounds.append(bkg)

                if plotting == True:
                    drz_obs.plot_psf_fit()
                    plt.show()

                    plt.imshow(residuals_from_model)
                    plt.title('Residual from 2D Smooth Background Model')
                    plt.colorbar()
                    plt.show()
                
    return smooth_stds, xshifts, yshifts, backgrounds



                


## Select level 2 and level 3 file paths

In [5]:
# SN 1983V
lvl2 = '/Users/sophiakressy/Desktop/Kavli/Summer_Project/SN1983V/HST/flc_lvl2/*_flc.fits'
lvl3 = '/Users/sophiakressy/Desktop/Kavli/Summer_Project/SN1983V/HST/drz_lvl3/*_drz.fits'
lvl3_single = '/Users/sophiakressy/Desktop/Kavli/Summer_Project/SN1983V/HST/drz_lvl3/F657N_drz.fits'

ra  = '3:33:31.6159' #unit=(u.hourangle,u.deg)) #icrs
dec = '-36:08:55.222' #unit=(u.hourangle,u.deg)) #icrs


### grab files

In [6]:
files = glob.glob(lvl2) #lvl2
files

filter_arr = []
file_arr = []
for flc in files:
    
    input_file= flc
    tmp = fits.open(input_file)
    fltr = tmp[0].header['FILTER']
    filter_arr.append(fltr)
    file_arr.append(input_file)
    
print(filter_arr)
print(file_arr)

filter_arr

['F657N', 'F336W', 'F555W', 'F657N', 'F555W', 'F814W', 'F275W', 'F657N', 'F657N', 'F814W', 'F438W', 'F275W', 'F438W', 'F814W', 'F275W', 'F336W', 'F336W', 'F555W', 'F438W']
['/Users/sophiakressy/Desktop/Kavli/Summer_Project/SN1983V/HST/flc_lvl2/F657N_if0404y2q_flc.fits', '/Users/sophiakressy/Desktop/Kavli/Summer_Project/SN1983V/HST/flc_lvl2/F336W_idxr09yuq_flc.fits', '/Users/sophiakressy/Desktop/Kavli/Summer_Project/SN1983V/HST/flc_lvl2/F555W_idxr09ztq_flc.fits', '/Users/sophiakressy/Desktop/Kavli/Summer_Project/SN1983V/HST/flc_lvl2/F657N_if0404xyq_flc.fits', '/Users/sophiakressy/Desktop/Kavli/Summer_Project/SN1983V/HST/flc_lvl2/F555W_idxr09z0q_flc.fits', '/Users/sophiakressy/Desktop/Kavli/Summer_Project/SN1983V/HST/flc_lvl2/F814W_idxr09z2q_flc.fits', '/Users/sophiakressy/Desktop/Kavli/Summer_Project/SN1983V/HST/flc_lvl2/F275W_idxr09ywq_flc.fits', '/Users/sophiakressy/Desktop/Kavli/Summer_Project/SN1983V/HST/flc_lvl2/F657N_if0404xxq_flc.fits', '/Users/sophiakressy/Desktop/Kavli/Summer_P

['F657N',
 'F336W',
 'F555W',
 'F657N',
 'F555W',
 'F814W',
 'F275W',
 'F657N',
 'F657N',
 'F814W',
 'F438W',
 'F275W',
 'F438W',
 'F814W',
 'F275W',
 'F336W',
 'F336W',
 'F555W',
 'F438W']

## Get Centroid

In [7]:
f = fits.open(lvl3_single)
data = f[1].data
w = WCS(f[1].header)

initial_location = SkyCoord(ra,dec,unit=(u.hourangle,u.deg)) #icrs
print(initial_location)

x_init, y_init = wcs.utils.skycoord_to_pixel(initial_location,w)
print(x_init, y_init)

x_cent, y_cent = centroid_sources(data, x_init, y_init, box_size=3, centroid_func=centroid_2dg)
print(x_cent,y_cent)

centered_location = wcs.utils.pixel_to_skycoord(x_cent[0],y_cent[0],w)
print(centered_location)

sn_location = centered_location
print(sn_location)

<SkyCoord (ICRS): (ra, dec) in deg
    (53.38173292, -36.14867278)>
1114.0631975164529 657.5237768768854
[1114.53758888] [657.83085591]
<SkyCoord (ICRS): (ra, dec) in deg
    (53.38173872, -36.14866869)>
<SkyCoord (ICRS): (ra, dec) in deg
    (53.38173872, -36.14866869)>


### Create a PSF using SPACE_PHOT in these DRZ tweakback images


# MIRI F770W

In [8]:
files = space_phot.util.filter_dict_from_list(glob.glob(lvl2),sn_location)['F657N']
files

filter_arr = []
file_arr = []
for flc in files:
    
    input_file= flc
    tmp = fits.open(input_file)
    fltr = tmp[0].header['FILTER']
    filter_arr.append(fltr)
    file_arr.append(input_file)
    
print(filter_arr)
print(file_arr)

filter = 'F657N'

indices = [i for i, x in enumerate(filter_arr) if x == filter]
filters = np.array(filter_arr)
files = np.array(file_arr)

tmp = np.array(indices)
filters[tmp]
input_flcs = files[tmp]
input_flcs = input_flcs.tolist()
input_flcs

drz_files = glob.glob(lvl3)
drz_files

filter_arr = []
file_arr = []
for flc in drz_files:
    
    input_file= flc
    tmp = fits.open(input_file)
    fltr = tmp[0].header['FILTER']
    filter_arr.append(fltr)
    file_arr.append(input_file)
    
print(filter_arr)
print(file_arr)

indices = [i for i, x in enumerate(filter_arr) if x == filter]
filters = np.array(filter_arr)
files = np.array(file_arr)

tmp = np.array(indices)
filters[tmp]
input_drzs = files[tmp]
input_drzs = input_drzs.tolist()

['F657N', 'F657N', 'F657N', 'F657N']
['/Users/sophiakressy/Desktop/Kavli/Summer_Project/SN1983V/HST/flc_lvl2/F657N_if0404y2q_flc.fits', '/Users/sophiakressy/Desktop/Kavli/Summer_Project/SN1983V/HST/flc_lvl2/F657N_if0404xyq_flc.fits', '/Users/sophiakressy/Desktop/Kavli/Summer_Project/SN1983V/HST/flc_lvl2/F657N_if0404xxq_flc.fits', '/Users/sophiakressy/Desktop/Kavli/Summer_Project/SN1983V/HST/flc_lvl2/F657N_if0404y0q_flc.fits']
['F438W', 'F814W', 'F336W', 'F555W', 'F657N', 'F275W']
['/Users/sophiakressy/Desktop/Kavli/Summer_Project/SN1983V/HST/drz_lvl3/F438W_drz.fits', '/Users/sophiakressy/Desktop/Kavli/Summer_Project/SN1983V/HST/drz_lvl3/F814W_drz.fits', '/Users/sophiakressy/Desktop/Kavli/Summer_Project/SN1983V/HST/drz_lvl3/F336W_drz.fits', '/Users/sophiakressy/Desktop/Kavli/Summer_Project/SN1983V/HST/drz_lvl3/F555W_drz.fits', '/Users/sophiakressy/Desktop/Kavli/Summer_Project/SN1983V/HST/drz_lvl3/F657N_drz.fits', '/Users/sophiakressy/Desktop/Kavli/Summer_Project/SN1983V/HST/drz_lvl3/F27

### Run PSF fit
* only need to run this once per filter

In [12]:
jwst_flc_obs = space_phot.observation2(input_flcs)
jwst_drz_obs = space_phot.observation3(input_drzs[0])
print(input_flcs)
print(input_drzs[0])
# has attributes .upper_limit, .psf_photometry, .create_psf_subtracted, .aperature_photometry, .plant_psf

# input_drzs[0] # use this as data and error #st_obs,st_obs3,sky_location,psf_width=25
# psf_drz = space_phot.get_hst3_psf(jwst_flc_obs, jwst_drz_obs, sn_location, psf_width=25) # this uses photutils to find PSF
psf_drz = space_phot.get_hst_psf(jwst_flc_obs, jwst_drz_obs, sn_location, num_psfs=4) # this uses photutils to find PSF

# plt.imshow(psf_drz.data)
# plt.show()

I am reading the header
['/Users/sophiakressy/Desktop/Kavli/Summer_Project/SN1983V/HST/flc_lvl2/F657N_if0404y2q_flc.fits', '/Users/sophiakressy/Desktop/Kavli/Summer_Project/SN1983V/HST/flc_lvl2/F657N_if0404xyq_flc.fits', '/Users/sophiakressy/Desktop/Kavli/Summer_Project/SN1983V/HST/flc_lvl2/F657N_if0404xxq_flc.fits', '/Users/sophiakressy/Desktop/Kavli/Summer_Project/SN1983V/HST/flc_lvl2/F657N_if0404y0q_flc.fits']
/Users/sophiakressy/Desktop/Kavli/Summer_Project/SN1983V/HST/drz_lvl3/F657N_drz.fits


TypeError: get_hst_psf() got an unexpected keyword argument 'num_psfs'

### Optimize PSF
* I recommend not setting 'plotting' == True if iterations are larger than 10!

In [ ]:
stddevs, x_shift, y_shift, back_ground = optimize_psf(drz_obs=jwst_drz_obs, drz_psf=psf_drz, sn_loc=sn_location, fwidth=11,
                                                      xshift_range=[-2,2], yshift_range=[-2,2], background_range=[2.33,2.33],
                                                      nxs=10, nys=10, nbkg=1, plotting=False)

# What different attributes does psf_drz contain? .data 
# drz_obs contains .psf_photometry

# Best Fit: xshift = 0.73
# yshift = -0.33
# Background = 2.33

#### Plot standard deviation for x_shift, y_shift parameter space
* plots the standard deviation for each iteration of the residual of smoothed background for pixel space around SN target
* does not address background, so need to do that manually?
* Change 's' marker size to fill space inbetween.
* Want to minimize the standard deviation, so look for global minimum in color map.
* If you aren't seeing a clear minimum in both x & y, expand search range above ^

In [ ]:
# plot 2D array, x v. y with color as std
plt.scatter(x_shift, y_shift, c=stddevs, s=800, marker='s') # s is a size of marker 
plt.viridis()
plt.xlabel('X Shift (pixels)')
plt.ylabel('Y Shift (pixels)')
plt.colorbar(label='Standard Deviation parameter')


#### Grab indices where stddev is minimum
* works for multiple varying parameters
* also plot the standard deviation as a function of the iterations (visual check)

In [ ]:
min_ind = np.argmin(stddevs)
print('Optimized Parameters: ')
print('Standard Deviation of 2D Smoothed Background: ', stddevs[min_ind]) 
print('x_shift: ', x_shift[min_ind],'y_shift: ', y_shift[min_ind])
print('background: ',back_ground[min_ind])

plt.plot(range(len(stddevs)), stddevs, marker='o', linestyle='none')
plt.xlabel('Run Index')
plt.ylabel('stddev(2D smooth bkgd model)')

#### Re-run with optimized parameters

In [ ]:
# stddevs, x_shift, y_shift, back_ground = optimize_psf(drz_obs=jwst_drz_obs, drz_psf=psf_drz, sn_loc=sn_location, fwidth=11,
#                                                       xshift_range=[x_shift[min_ind],x_shift[min_ind]], 
#                                                       yshift_range=[y_shift[min_ind],y_shift[min_ind]], 
#                                                       background_range=[back_ground[min_ind],back_ground[min_ind]],
#                                                       nxs=1, nys=1, nbkg=1, plotting=True)

# Optimized params
stddevs, x_shift, y_shift, back_ground = optimize_psf(drz_obs=jwst_drz_obs, drz_psf=psf_drz, sn_loc=sn_location, fwidth=11,
                                                      xshift_range=[0.666,0.666], 
                                                      yshift_range=[-0.22,-0.22], 
                                                      background_range=[2.33,2.33],
                                                      nxs=1, nys=1, nbkg=1, plotting=True)

In [ ]:
jwst_drz_obs.psf_photometry(psf_drz,sn_location,bounds={'flux':[-30000,40000],
                            'centroid':[-1,1],                              
                            'bkg':[-10,10]},
                            fit_width=11,
                            background = 2.33,
                            fit_bkg=False,
                            fit_centroid=False, 
                            fit_flux=True,
                            centroidx_shift=0.666,
                            centroidy_shift=-0.22,
                            psf_method='nest',
                            center_weight=25,npoints=200)



## Test with PhotUtils error comparison

In [ ]:
from photutils.psf import PSFPhotometry
from astropy.io import fits
from astropy.table import Table
from astropy.nddata import NDData, StdDevUncertainty
from photutils.psf import IntegratedGaussianPRF
from astropy.modeling.fitting import LevMarLSQFitter
from astropy.stats import gaussian_sigma_to_fwhm
import matplotlib.pyplot as plt


In [ ]:
print(input_drzs[0])
# F770  = fits.open(input_drzs[0])
# data  = F770['SCI',1].data
# error = F770['ERR',1].data
# f770wcs = WCS(F770[1].header)
# print(sn_location)
# # x_cent,y_cent

In [ ]:
# import numpy as np
# from astropy.nddata import NDData, StdDevUncertainty
# from astropy.modeling.fitting import LevMarLSQFitter
# from astropy.table import Table
# from photutils.psf import IntegratedGaussianPRF
# from astropy.stats import gaussian_sigma_to_fwhm
# from photutils.psf import 


# def run_psf_photometry(image_data, error_data, xpos, ypos, fwhm=3.0, cutout_radius=5):
#     """
#     Perform PSF photometry on a single source and return
#     fitted flux, flux uncertainty, and S/N.

#     Parameters
#     ----------
#     image_data : 2D array
#         Background-subtracted image.
#     error_data : 2D array
#         1-sigma uncertainty map (same shape as image_data).
#     xpos, ypos : float
#         Approximate source coordinates (pixels).
#     fwhm : float
#         PSF FWHM in pixels (default: 3.0).
#     cutout_radius : int
#         Half-size of fitting cutout region in pixels (default: 5).

#     Returns
#     -------
#     results : dict
#         Dictionary with x_fit, y_fit, flux, flux_err, SNR.
#     """
#     # Create cutout region
#     x_min = int(xpos) - cutout_radius
#     x_max = int(xpos) + cutout_radius + 1
#     y_min = int(ypos) - cutout_radius
#     y_max = int(ypos) + cutout_radius + 1

#     data_cutout = image_data[y_min:y_max, x_min:x_max]
#     error_cutout = error_data[y_min:y_max, x_min:x_max]

#     # Coordinate grid
#     yy, xx = np.mgrid[y_min:y_max, x_min:x_max]

#     # Define PSF model
#     sigma = fwhm / gaussian_sigma_to_fwhm
#     psf_model = IntegratedGaussianPRF(sigma=sigma)
#     psf_model.sigma.fixed = True
#     psf_model.x_0 = xpos
#     psf_model.y_0 = ypos

#     # Rough flux guess: sum of small cutout
#     flux_guess = np.sum(data_cutout)
#     psf_model.flux = flux_guess

#     # Fit model
#     fitter = LevMarLSQFitter()
#     fitted_model = fitter(psf_model, xx, yy, data_cutout, weights=1.0 / error_cutout)

#     # Get best-fit parameters
#     x_fit = fitted_model.x_0.value
#     y_fit = fitted_model.y_0.value
#     flux_fit = fitted_model.flux.value

#     # Get flux uncertainty from covariance matrix if available
#     flux_err = np.nan
#     if fitter.fit_info is not None and "param_cov" in fitter.fit_info:
#         cov = fitter.fit_info["param_cov"]
#         flux_var = cov[-1, -1]  # last parameter is flux
#         if flux_var > 0:
#             flux_err = np.sqrt(flux_var)

#     snr = flux_fit / flux_err if np.isfinite(flux_err) else np.nan

#     return {
#         "x_fit": x_fit,
#         "y_fit": y_fit,
#         "flux": flux_fit,
#         "flux_err": flux_err,
#         "SNR": snr,
#     }


# # -----------------------------
# # Example usage
# # -----------------------------
# background_value = 2.33
# image_data = fits.open(input_drzs[0])['SCI', 1].data - background_value
# error_data = fits.open(input_drzs[0])['ERR', 1].data

# result = run_psf_photometry(image_data, error_data, xpos=x_cent, ypos=y_cent)

# print(f"Fitted position: ({result['x_fit']:.2f}, {result['y_fit']:.2f})")
# print(f"Flux: {result['flux']:.4f} ± {result['flux_err']:.4f}")
# print(f"S/N: {result['SNR']:.2f}")


In [ ]:
# -----------------------------
# Load FITS image and error map
# -----------------------------
background_value = 2.33  # <-- replace with your known background

with fits.open(input_drzs[0]) as F770:
    image_data = F770['SCI',1].data
    error_data = F770['ERR',1].data

# Subtract background before NDData creation
image_data = image_data - background_value

# ndata = NDData(data=image_data, uncertainty=error_data)
ndata = NDData(data=image_data, uncertainty=StdDevUncertainty(error_data))


# -----------------------------
# Define the PSF model
# -----------------------------
fwhm = 4.0  # pixels
sigma = fwhm / gaussian_sigma_to_fwhm

psf_model = IntegratedGaussianPRF(sigma=sigma)
psf_model.sigma.fixed = True   # fix PSF shape

# Supply known coordinates
xpos, ypos = x_cent,y_cent  # <-- replace with your source position

psf_model.x_0 = xpos
psf_model.y_0 = ypos
psf_model.x_0.fixed = True
psf_model.y_0.fixed = True

# -----------------------------
# Set up photometry
# -----------------------------
fitter = LevMarLSQFitter()

photometry = PSFPhotometry(psf_model=psf_model, fitter=fitter, fit_shape=(11,11), aperture_radius=fwhm)

# -----------------------------
# Run photometry
# -----------------------------

flux_guess = np.sum(image_data[
    int(ypos)-3:int(ypos)+4,
    int(xpos)-3:int(xpos)+4
])

init_params = Table()
init_params['x_0'] = [xpos]
init_params['y_0'] = [ypos]
init_params['flux_0'] = [flux_guess]

result_tab = photometry(ndata, init_params=init_params)

print(result_tab.keys())

flux = result_tab['flux_fit']
flux_err = result_tab['flux_err']

# Get the model image (same shape as your data)
model_image = photometry.make_model_image(image_data.shape)

# Get the residual image (observed - model)
residual_nddata = photometry.make_residual_image(ndata)
# Convert to pure numpy array safely
residual_image = np.asarray(residual_nddata.data)
print(flux)
print(flux_err)

# Image pixel location of SN 83V taken from DS9
x = round(1139)
y = round(1244)
# Crop image with indexing to focus on SN -- for cropping, [y,x]
window = 20
y0 = y-window
y1 = y+window
x0 = x-window
x1 = x+window

# Crop region (same as before)
crop_small = (slice(y0, y1), slice(x0, x1))

# Set figure space
# fig = plt.figure(figsize=(8,8))
# ax = plt.subplot(projection = wcs)

# plot image in log space
# miri770 = ax.imshow(miri_770_data[crop_small], norm='log', vmax=100, vmin=0.1)


fig, axs = plt.subplots(1, 3, figsize=(12, 4))

# Original image
axs[0].imshow(image_data[crop_small], origin='lower', cmap='viridis')
axs[0].set_title("Original Image")

# Model image
axs[1].imshow(model_image[crop_small], origin='lower', cmap='viridis')
axs[1].set_title("Model Image")

# Residual image with symmetric color scale
vmax = np.nanstd(residual_image)
im = axs[2].imshow(residual_image[crop_small], origin='lower', cmap='RdBu_r',
                   vmin=-15, vmax=15)
axs[2].set_title("Residual Image")
plt.colorbar(im, ax=axs[2], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()


In [ ]:
#psf FWHM for MIRI 770-2100 = 0.269" - 0.674" or 2.445 - 6.127 pixels = avg 0.47" or 4.2pix = median 0.32" or 3pix
psf_width = 0.3 # pixels
fwidth = 11
psfphot = PSFPhotometry(psf_drz, fwidth, aperture_radius=psf_width)



psfphot = PSFPhotometry(psf_drz, psf_width, finder=daofind,
                        aperture_radius=psf_width, localbkg_estimator=localbkg_estimator)

# pass 
phot = psfphot(data, error=error,init_params=pos)
print(psfphot.finder_results) 

In [ ]:
print(psfphot.__dict__.keys())
print(psfphot.fit_params)

In [ ]:
phot = psfphot(data, error=error,init_params=pos)
print(psfphot.finder_results) 

In [ ]:
# from photutils.detection import DAOStarFinder
# from photutils.background import Background2D, MedianBackground
# from photutils.background import LocalBackground
# import numpy as np




# class MyLocalMedianBackground(LocalBackground):
#     def __init__(self, inner_radius, outer_radius):
#         super().__init__(inner_radius, outer_radius)
#     def __call__(self, data):
#         return np.median(data)

# # Example radii (adjust as needed for your data)
# localbkg_estimator = MyLocalMedianBackground(inner_radius=5, outer_radius=8)

# bkg_estimator = MedianBackground()
# bkg = Background2D(data, box_size=50, filter_size=3, bkg_estimator=bkg_estimator)
# data_bkgsub = data - bkg.background

# # psf_drz.data = psf model data

# background = 2.33
# xshift=0.666
# yshift=-0.22
# psf_width=5

# yi,xi = skycoord_to_pixel(sn_location,f770wcs)
# xi+=xshift
# yi+=yshift
# center = [xi+xshift,yi+yshift]
# centers = []
# centers.append(center)

# pos = Table(np.atleast_2d(centers),names=['y_0','x_0'])

# daofind = DAOStarFinder(threshold=1,fwhm=2,xycoords=np.array([pos['x_0'],pos['y_0']]).T)
# sources = daofind(data_bkgsub)
# print(sources)
# # psfphot = PSFPhotometry(self.psf_model_list[0], psf_width, finder=daofind,
# #                     aperture_radius=psf_width,localbkg_estimator=localbkg_estimator)

# psfphot = PSFPhotometry(psf_drz, psf_width, finder=daofind,
#                         aperture_radius=psf_width, localbkg_estimator=localbkg_estimator)

# phot = psfphot(data, error=error,init_params=pos)
# print(psfphot.finder_results) 

#### Print magnitude and error

In [ ]:
print(jwst_drz_obs.psf_result.phot_cal_table)
print(jwst_drz_obs.psf_result.phot_cal_table['mag','magerr'])
print(jwst_drz_obs.psf_result.phot_cal_table['flux'])
print(jwst_drz_obs.psf_result.phot_cal_table['fluxerr'])

mag_ab  = jwst_drz_obs.psf_result.phot_cal_table['mag'][0]
mag_err = jwst_drz_obs.psf_result.phot_cal_table['magerr'][0]

#### Calculate Flux from magnitudes found

In [ ]:
# Upper Limit
# print('Upper Limit: ', jwst_drz_obs.upper_limit(nsigma=5))

flux_jy = 10**(-0.4 * (mag_ab + 48.6)) * 1e23
flux_err_jy = 0.4 * np.log(10) * flux_jy * mag_err
print('FLUX [Jy] = ', flux_jy)
print('FLUX ERROR [Jy] = ', flux_err_jy)


# MIRI F1000W

In [ ]:
files = space_phot.util.filter_dict_from_list(glob.glob(lvl2),sn_location)['F1000W']
files

filter_arr = []
file_arr = []
for flc in files:
    
    input_file= flc
    tmp = fits.open(input_file)
    fltr = tmp[0].header['FILTER']
    filter_arr.append(fltr)
    file_arr.append(input_file)
    
print(filter_arr)
print(file_arr)

filter = 'F1000W'

indices = [i for i, x in enumerate(filter_arr) if x == filter]
filters = np.array(filter_arr)
files = np.array(file_arr)

tmp = np.array(indices)
filters[tmp]
input_flcs = files[tmp]
input_flcs = input_flcs.tolist()
input_flcs

drz_files = glob.glob(lvl3)
drz_files

filter_arr = []
file_arr = []
for flc in drz_files:
    
    input_file= flc
    tmp = fits.open(input_file)
    fltr = tmp[0].header['FILTER']
    filter_arr.append(fltr)
    file_arr.append(input_file)
    
print(filter_arr)
print(file_arr)

indices = [i for i, x in enumerate(filter_arr) if x == filter]
filters = np.array(filter_arr)
files = np.array(file_arr)

tmp = np.array(indices)
filters[tmp]
input_drzs = files[tmp]
input_drzs = input_drzs.tolist()

### Run PSF fit

In [ ]:
jwst_flc_obs = space_phot.observation2(input_flcs)
jwst_drz_obs = space_phot.observation3(input_drzs[0])

psf_drz = space_phot.get_jwst3_psf(jwst_flc_obs, jwst_drz_obs, sn_location, num_psfs=4)
plt.imshow(psf_drz.data)
plt.show()

In [ ]:
stddevs, x_shift, y_shift, back_ground = optimize_psf(drz_obs=jwst_drz_obs, drz_psf=psf_drz, sn_loc=sn_location, fwidth=11,
                                                      xshift_range=[-2,2], yshift_range=[-2,2], background_range=[0.77,0.77],
                                                      nxs=10, nys=10, nbkg=1, plotting=False)

# Best Fit: X shift = 0.42, Y shift = -0.4, background = 0.77

In [ ]:
min_ind = np.argmin(stddevs)
print('Optimized Parameters: ')
print('Standard Deviation of 2D Smoothed Background: ', stddevs[min_ind]) 
print('x_shift: ', x_shift[min_ind],'y_shift: ', y_shift[min_ind])
print('background: ',back_ground[min_ind])

plt.plot(range(len(stddevs)), stddevs, marker='o', linestyle='none')
plt.xlabel('Run Index')
plt.ylabel('stddev(2D smooth bkgd model)')

In [ ]:
# stddevs, x_shift, y_shift, back_ground = optimize_psf(drz_obs=jwst_drz_obs, drz_psf=psf_drz, sn_loc=sn_location, fwidth=11,
#                                                       xshift_range=[x_shift[min_ind],x_shift[min_ind]], 
#                                                       yshift_range=[y_shift[min_ind],y_shift[min_ind]], 
#                                                       background_range=[back_ground[min_ind],back_ground[min_ind]],
#                                                       nxs=1, nys=1, nbkg=1, plotting=True)

stddevs, x_shift, y_shift, back_ground = optimize_psf(drz_obs=jwst_drz_obs, drz_psf=psf_drz, sn_loc=sn_location, fwidth=11,
                                                      xshift_range=[0.222,0.222], 
                                                      yshift_range=[-0.222,-0.222], 
                                                      background_range=[0.77, 0.77],
                                                      nxs=1, nys=1, nbkg=1, plotting=True)

#### Print magnitude and error

In [ ]:
print(jwst_drz_obs.psf_result.phot_cal_table)
print(jwst_drz_obs.psf_result.phot_cal_table['mag','magerr'])

mag_ab  = jwst_drz_obs.psf_result.phot_cal_table['mag'][0]
mag_err = jwst_drz_obs.psf_result.phot_cal_table['magerr'][0]

#### Calculate Flux

In [ ]:
# Upper Limit
# print('Upper Limit: ', jwst_drz_obs.upper_limit(nsigma=5))

flux_jy = 10**(-0.4 * (mag_ab + 48.6)) * 1e23
flux_err_jy = 0.4 * np.log(10) * flux_jy * mag_err
print('FLUX [Jy] = ', flux_jy)
print('FLUX ERROR [Jy] = ', flux_err_jy)


# MIRI F1130W

## Since 1130 is misaligned, assign new centroid. 
## SKIP IF PROCEEDING TO F2100W

In [ ]:
f = fits.open(lvl3_single_1130)
data = f[1].data
w = WCS(f[1].header)

initial_location = SkyCoord(ra1130,dec1130,unit=(u.hourangle,u.deg)) #icrs
print(initial_location)

x_init, y_init = wcs.utils.skycoord_to_pixel(initial_location,w)
print(x_init, y_init)

x_cent, y_cent = centroid_sources(data, x_init, y_init, box_size=3, centroid_func=centroid_2dg)
print(x_cent,y_cent)

centered_location = wcs.utils.pixel_to_skycoord(x_cent[0],y_cent[0],w)
print(centered_location)

sn_location = centered_location
print(sn_location)

In [ ]:
files = space_phot.util.filter_dict_from_list(glob.glob(lvl2),sn_location)['F1130W']
files

filter_arr = []
file_arr = []
for flc in files:
    
    input_file= flc
    tmp = fits.open(input_file)
    fltr = tmp[0].header['FILTER']
    filter_arr.append(fltr)
    file_arr.append(input_file)
    
print(filter_arr)
print(file_arr)

filter = 'F1130W'

indices = [i for i, x in enumerate(filter_arr) if x == filter]
filters = np.array(filter_arr)
files = np.array(file_arr)

tmp = np.array(indices)
filters[tmp]
input_flcs = files[tmp]
input_flcs = input_flcs.tolist()
input_flcs

drz_files = glob.glob(lvl3)

drz_files

filter_arr = []
file_arr = []
for flc in drz_files:
    
    input_file= flc
    tmp = fits.open(input_file)
    fltr = tmp[0].header['FILTER']
    filter_arr.append(fltr)
    file_arr.append(input_file)
    
print(filter_arr)
print(file_arr)

indices = [i for i, x in enumerate(filter_arr) if x == filter]
filters = np.array(filter_arr)
files = np.array(file_arr)

tmp = np.array(indices)
filters[tmp]
input_drzs = files[tmp]
input_drzs = input_drzs.tolist()
input_drzs


In [ ]:
jwst_flc_obs = space_phot.observation2(input_flcs)
jwst_drz_obs = space_phot.observation3(input_drzs[0])
psf_drz = space_phot.get_jwst3_psf(jwst_flc_obs, jwst_drz_obs, sn_location, num_psfs=4)
plt.imshow(psf_drz.data)
plt.show()

In [ ]:
stddevs, x_shift, y_shift, back_ground = optimize_psf(drz_obs=jwst_drz_obs, drz_psf=psf_drz, sn_loc=sn_location, fwidth=11,
                                                      xshift_range=[-2,2], yshift_range=[-2,2], background_range=[2.628,2.628],
                                                      nxs=10, nys=10, nbkg=1, plotting=False)

# Best fit params X shift = 0.55, Y shift = -0.45, bkg = 2.628


In [ ]:
# plot 2D array, x v. y with color as std
plt.scatter(x_shift, y_shift, c=stddevs, s=800, marker='s') # s is a size of marker 
plt.viridis()
plt.xlabel('X Shift (pixels)')
plt.ylabel('Y Shift (pixels)')
plt.colorbar(label='Standard Deviation parameter')

In [ ]:
min_ind = np.argmin(stddevs)
print('Optimized Parameters: ')
print('Standard Deviation of 2D Smoothed Background: ', stddevs[min_ind]) 
print('x_shift: ', x_shift[min_ind],'y_shift: ', y_shift[min_ind])
print('background: ',back_ground[min_ind])

plt.plot(range(len(stddevs)), stddevs, marker='o', linestyle='none')
plt.xlabel('Run Index')
plt.ylabel('stddev(2D smooth bkgd model)')

In [ ]:
# stddevs, x_shift, y_shift, back_ground = optimize_psf(drz_obs=jwst_drz_obs, drz_psf=psf_drz, sn_loc=sn_location, fwidth=11,
#                                                       xshift_range=[x_shift[min_ind],x_shift[min_ind]], 
#                                                       yshift_range=[y_shift[min_ind],y_shift[min_ind]], 
#                                                       background_range=[back_ground[min_ind],back_ground[min_ind]],
#                                                       nxs=1, nys=1, nbkg=1, plotting=True)

stddevs, x_shift, y_shift, back_ground = optimize_psf(drz_obs=jwst_drz_obs, drz_psf=psf_drz, sn_loc=sn_location, fwidth=11,
                                                      xshift_range=[0.6666,0.6666], 
                                                      yshift_range=[-0.222,-0.222], 
                                                      background_range=[2.628, 2.628],
                                                      nxs=1, nys=1, nbkg=1, plotting=True)

#### Print magnitude and error

In [ ]:
print(jwst_drz_obs.psf_result.phot_cal_table)
print(jwst_drz_obs.psf_result.phot_cal_table['mag','magerr'])

mag_ab  = jwst_drz_obs.psf_result.phot_cal_table['mag'][0]
mag_err = jwst_drz_obs.psf_result.phot_cal_table['magerr'][0]

#### Calculate Flux

In [ ]:
# Upper Limit
# print('Upper Limit: ', jwst_drz_obs.upper_limit(nsigma=5))

flux_jy = 10**(-0.4 * (mag_ab + 48.6)) * 1e23
flux_err_jy = 0.4 * np.log(10) * flux_jy * mag_err
print('FLUX [Jy] = ', flux_jy)
print('FLUX ERROR [Jy] = ', flux_err_jy)


# MIRI F2100W

In [ ]:
files = space_phot.util.filter_dict_from_list(glob.glob(lvl2),sn_location)['F2100W']
files

filter_arr = []
file_arr = []
for flc in files:
    
    input_file= flc
    tmp = fits.open(input_file)
    fltr = tmp[0].header['FILTER']
    filter_arr.append(fltr)
    file_arr.append(input_file)
    
print(filter_arr)
print(file_arr)

filter = 'F2100W'

indices = [i for i, x in enumerate(filter_arr) if x == filter]
filters = np.array(filter_arr)
files = np.array(file_arr)

tmp = np.array(indices)
filters[tmp]
input_flcs = files[tmp]
input_flcs = input_flcs.tolist()
input_flcs

drz_files = glob.glob(lvl3)
drz_files

filter_arr = []
file_arr = []
for flc in drz_files:
    
    input_file= flc
    tmp = fits.open(input_file)
    fltr = tmp[0].header['FILTER']
    filter_arr.append(fltr)
    file_arr.append(input_file)
    
print(filter_arr)
print(file_arr)

indices = [i for i, x in enumerate(filter_arr) if x == filter]
filters = np.array(filter_arr)
files = np.array(file_arr)

tmp = np.array(indices)
filters[tmp]
input_drzs = files[tmp]
input_drzs = input_drzs.tolist()

### Run PSF fit

In [ ]:
jwst_flc_obs = space_phot.observation2(input_flcs)
jwst_drz_obs = space_phot.observation3(input_drzs[0])

psf_drz = space_phot.get_jwst3_psf(jwst_flc_obs, jwst_drz_obs, sn_location, num_psfs=4)
plt.imshow(psf_drz.data)
plt.show()

In [ ]:
stddevs, x_shift, y_shift, back_ground = optimize_psf(drz_obs=jwst_drz_obs, drz_psf=psf_drz, sn_loc=sn_location, fwidth=11,
                                                      xshift_range=[-2,2], yshift_range=[-2,2], background_range=[4.55,4.55],
                                                      nxs=10, nys=10, nbkg=1, plotting=False)

# Best Fit: X Shift = -0.22, Y shift = -0.11, bkg = 4.55

In [ ]:
# plot 2D array, x v. y with color as std
plt.scatter(x_shift, y_shift, c=stddevs, s=800, marker='s') # s is a size of marker 
plt.viridis()
plt.xlabel('X Shift (pixels)')
plt.ylabel('Y Shift (pixels)')
plt.colorbar(label='Standard Deviation parameter')


In [ ]:
min_ind = np.argmin(stddevs)
print('Optimized Parameters: ')
print('Standard Deviation of 2D Smoothed Background: ', stddevs[min_ind]) 
print('x_shift: ', x_shift[min_ind],'y_shift: ', y_shift[min_ind])
print('background: ',back_ground[min_ind])

plt.plot(range(len(stddevs)), stddevs, marker='o', linestyle='none')
plt.xlabel('Run Index')
plt.ylabel('stddev(2D smooth bkgd model)')

In [ ]:
# stddevs, x_shift, y_shift, back_ground = optimize_psf(drz_obs=jwst_drz_obs, drz_psf=psf_drz, sn_loc=sn_location, fwidth=11,
#                                                       xshift_range=[x_shift[min_ind],x_shift[min_ind]], 
#                                                       yshift_range=[y_shift[min_ind],y_shift[min_ind]], 
#                                                       background_range=[back_ground[min_ind],back_ground[min_ind]],
#                                                       nxs=1, nys=1, nbkg=1, plotting=True)

stddevs, x_shift, y_shift, back_ground = optimize_psf(drz_obs=jwst_drz_obs, drz_psf=psf_drz, sn_loc=sn_location, fwidth=11,
                                                      xshift_range=[-0.222,-0.222], 
                                                      yshift_range=[-0.22,-0.22], 
                                                      background_range=[4.55, 4.55],
                                                      nxs=1, nys=1, nbkg=1, plotting=True)

#### Print magnitude and error

In [ ]:
print(jwst_drz_obs.psf_result.phot_cal_table)
print(jwst_drz_obs.psf_result.phot_cal_table['mag','magerr'])

mag_ab  = jwst_drz_obs.psf_result.phot_cal_table['mag'][0]
mag_err = jwst_drz_obs.psf_result.phot_cal_table['magerr'][0]

#### Calculate Flux

In [ ]:
# Upper Limit
# print('Upper Limit: ', jwst_drz_obs.upper_limit(nsigma=5))

flux_jy = 10**(-0.4 * (mag_ab + 48.6)) * 1e23
flux_err_jy = 0.4 * np.log(10) * flux_jy * mag_err
print('FLUX [Jy] = ', flux_jy)
print('FLUX ERROR [Jy] = ', flux_err_jy)


## ----------------------------------------------------------------------------------------
## ----------------------------------------------------------------------------------------

# NIRCAM F200W

### Grab files

In [ ]:
# SN 1983V
lvl2 = '/Users/sophiakressy/Desktop/Kavli/Summer_Project/SN1983V/lvl_2_nircam/*_cal.fits'
lvl3 = '/Users/sophiakressy/Desktop/Kavli/Summer_Project/SN1983V/lvl_3/*jhat_i2d.fits'
lvl3_single = '/Users/sophiakressy/Desktop/Kavli/Summer_Project/SN1983V/lvl_3/jw02107-o021_t003_nircam_clear-f200w_jhat_i2d.fits'

ra  = '3:33:31.6162' #unit=(u.hourangle,u.deg)) #icrs
dec = '-36:08:55.251' #unit=(u.hourangle,u.deg)) #icrs


In [ ]:
files = glob.glob(lvl2) #lvl2
files

filter_arr = []
file_arr = []
for flc in files:
    
    input_file= flc
    tmp = fits.open(input_file)
    fltr = tmp[0].header['FILTER']
    filter_arr.append(fltr)
    file_arr.append(input_file)
    
print(filter_arr)
print(file_arr)

filter_arr

### Centroid

In [ ]:
f = fits.open(lvl3_single)
data = f[1].data
w = WCS(f[1].header)

initial_location = SkyCoord(ra,dec,unit=(u.hourangle,u.deg)) #icrs
print(initial_location)

x_init, y_init = wcs.utils.skycoord_to_pixel(initial_location,w)
print(x_init, y_init)

x_cent, y_cent = centroid_sources(data, x_init, y_init, box_size=3, centroid_func=centroid_2dg)
print(x_cent,y_cent)

centered_location = wcs.utils.pixel_to_skycoord(x_cent[0],y_cent[0],w)
print(centered_location)

sn_location = centered_location
print(sn_location)

### Now run through filters

In [ ]:
files = space_phot.util.filter_dict_from_list(glob.glob(lvl2),sn_location)['F200W']
files

filter_arr = []
file_arr = []
for flc in files:
    
    input_file= flc
    tmp = fits.open(input_file)
    fltr = tmp[0].header['FILTER']
    filter_arr.append(fltr)
    file_arr.append(input_file)
    
print(filter_arr)
print(file_arr)

filter = 'F200W'

indices = [i for i, x in enumerate(filter_arr) if x == filter]
filters = np.array(filter_arr)
files = np.array(file_arr)

tmp = np.array(indices)
filters[tmp]
input_flcs = files[tmp]
input_flcs = input_flcs.tolist()
input_flcs

drz_files = glob.glob(lvl3)

drz_files

filter_arr = []
file_arr = []
for flc in drz_files:
    
    input_file= flc
    tmp = fits.open(input_file)
    fltr = tmp[0].header['FILTER']
    filter_arr.append(fltr)
    file_arr.append(input_file)
    
print(filter_arr)
print(file_arr)

indices = [i for i, x in enumerate(filter_arr) if x == filter]
filters = np.array(filter_arr)
files = np.array(file_arr)

tmp = np.array(indices)
filters[tmp]
input_drzs = files[tmp]
input_drzs = input_drzs.tolist()
input_drzs

input_flcs



#### Run PSF

In [ ]:
jwst_flc_obs = space_phot.observation2(input_flcs) 
jwst_drz_obs = space_phot.observation3(input_drzs[0])
psf_drz = space_phot.get_jwst3_psf(jwst_flc_obs, jwst_drz_obs, sn_location, num_psfs=4)
plt.imshow(psf_drz.data)
plt.show()

In [ ]:
stddevs, x_shift, y_shift, back_ground = optimize_psf(drz_obs=jwst_drz_obs, drz_psf=psf_drz, sn_loc=sn_location, fwidth=9,
                                                      xshift_range=[-2,2], yshift_range=[-2,2], background_range=[1.11,1.11],
                                                      nxs=10, nys=10, nbkg=1, plotting=False)

# x = 0.666
# y = 0.66
# background = 1.11

In [ ]:
# plot 2D array, x v. y with color as std
plt.scatter(x_shift, y_shift, c=stddevs, s=800, marker='s') # s is a size of marker 
plt.viridis()
plt.xlabel('X Shift (pixels)')
plt.ylabel('Y Shift (pixels)')
plt.colorbar(label='Standard Deviation parameter')


In [ ]:
min_ind = np.argmin(stddevs)
print('Optimized Parameters: ')
print('Standard Deviation of 2D Smoothed Background: ', stddevs[min_ind]) 
print('x_shift: ', x_shift[min_ind],'y_shift: ', y_shift[min_ind])
print('background: ',back_ground[min_ind])

plt.plot(range(len(stddevs)), stddevs, marker='o', linestyle='none')
plt.xlabel('Run Index')
plt.ylabel('stddev(2D smooth bkgd model)')

In [ ]:
stddevs, x_shift, y_shift, back_ground = optimize_psf(drz_obs=jwst_drz_obs, drz_psf=psf_drz, sn_loc=sn_location, fwidth=9,
                                                      xshift_range=[x_shift[min_ind],x_shift[min_ind]], 
                                                      yshift_range=[y_shift[min_ind],y_shift[min_ind]], 
                                                      background_range=[back_ground[min_ind],back_ground[min_ind]],
                                                      nxs=1, nys=1, nbkg=1, plotting=True)

#### Print magnitude and error

In [ ]:
print(jwst_drz_obs.psf_result.phot_cal_table)
print(jwst_drz_obs.psf_result.phot_cal_table['mag','magerr'])

mag_ab  = jwst_drz_obs.psf_result.phot_cal_table['mag'][0]
mag_err = jwst_drz_obs.psf_result.phot_cal_table['magerr'][0]

#### Calculate Flux

In [ ]:
# Upper Limit
# print('Upper Limit: ', jwst_drz_obs.upper_limit(nsigma=5))

flux_jy = 10**(-0.4 * (mag_ab + 48.6)) * 1e23
flux_err_jy = 0.4 * np.log(10) * flux_jy * mag_err
print('FLUX [Jy] = ', flux_jy)
print('FLUX ERROR [Jy] = ', flux_err_jy)


# NIRCAM F300M

In [ ]:
files = space_phot.util.filter_dict_from_list(glob.glob(lvl2),sn_location)['F300M']
files

filter_arr = []
file_arr = []
for flc in files:
    
    input_file= flc
    tmp = fits.open(input_file)
    fltr = tmp[0].header['FILTER']
    filter_arr.append(fltr)
    file_arr.append(input_file)
    
print(filter_arr)
print(file_arr)

filter = 'F300M'

indices = [i for i, x in enumerate(filter_arr) if x == filter]
filters = np.array(filter_arr)
files = np.array(file_arr)

tmp = np.array(indices)
filters[tmp]
input_flcs = files[tmp]
input_flcs = input_flcs.tolist()
input_flcs

drz_files = glob.glob(lvl3)

drz_files

filter_arr = []
file_arr = []
for flc in drz_files:
    
    input_file= flc
    tmp = fits.open(input_file)
    fltr = tmp[0].header['FILTER']
    filter_arr.append(fltr)
    file_arr.append(input_file)
    
print(filter_arr)
print(file_arr)

indices = [i for i, x in enumerate(filter_arr) if x == filter]
filters = np.array(filter_arr)
files = np.array(file_arr)

tmp = np.array(indices)
filters[tmp]
input_drzs = files[tmp]
input_drzs = input_drzs.tolist()
input_drzs

input_flcs



In [ ]:
jwst_flc_obs = space_phot.observation2(input_flcs) 
jwst_drz_obs = space_phot.observation3(input_drzs[0])
psf_drz = space_phot.get_jwst3_psf(jwst_flc_obs, jwst_drz_obs, sn_location, num_psfs=4)
plt.imshow(psf_drz.data)
plt.show()

In [ ]:
stddevs, x_shift, y_shift, back_ground = optimize_psf(drz_obs=jwst_drz_obs, drz_psf=psf_drz, sn_loc=sn_location, fwidth=11,
                                                      xshift_range=[-2,2], yshift_range=[-2,2], background_range=[0.55,0.55],
                                                      nxs=10, nys=10, nbkg=1, plotting=False)

# x = 0.55
# y = -0.44
# background = 0.55


In [ ]:
# plot 2D array, x v. y with color as std
plt.scatter(x_shift, y_shift, c=stddevs, s=800, marker='s') # s is a size of marker 
plt.viridis()
plt.xlabel('X Shift (pixels)')
plt.ylabel('Y Shift (pixels)')
plt.colorbar(label='Standard Deviation parameter')


In [ ]:
min_ind = np.argmin(stddevs)
print('Optimized Parameters: ')
print('Standard Deviation of 2D Smoothed Background: ', stddevs[min_ind]) 
print('x_shift: ', x_shift[min_ind],'y_shift: ', y_shift[min_ind])
print('background: ',back_ground[min_ind])

plt.plot(range(len(stddevs)), stddevs, marker='o', linestyle='none')
plt.xlabel('Run Index')
plt.ylabel('stddev(2D smooth bkgd model)')

In [ ]:
stddevs, x_shift, y_shift, back_ground = optimize_psf(drz_obs=jwst_drz_obs, drz_psf=psf_drz, sn_loc=sn_location, fwidth=11,
                                                      xshift_range=[x_shift[min_ind],x_shift[min_ind]], 
                                                      yshift_range=[y_shift[min_ind],y_shift[min_ind]], 
                                                      background_range=[back_ground[min_ind],back_ground[min_ind]],
                                                      nxs=1, nys=1, nbkg=1, plotting=True)

#### Print magnitude and error

In [ ]:
print(jwst_drz_obs.psf_result.phot_cal_table)
print(jwst_drz_obs.psf_result.phot_cal_table['mag','magerr'])

mag_ab  = jwst_drz_obs.psf_result.phot_cal_table['mag'][0]
mag_err = jwst_drz_obs.psf_result.phot_cal_table['magerr'][0]

#### Calculate Flux

In [ ]:
# Upper Limit
# print('Upper Limit: ', jwst_drz_obs.upper_limit(nsigma=5))

flux_jy = 10**(-0.4 * (mag_ab + 48.6)) * 1e23
flux_err_jy = 0.4 * np.log(10) * flux_jy * mag_err
print('FLUX [Jy] = ', flux_jy)
print('FLUX ERROR [Jy] = ', flux_err_jy)


# NIRCAM F335M

In [ ]:
files = space_phot.util.filter_dict_from_list(glob.glob(lvl2),sn_location)['F335M']
files

filter_arr = []
file_arr = []
for flc in files:
    
    input_file= flc
    tmp = fits.open(input_file)
    fltr = tmp[0].header['FILTER']
    filter_arr.append(fltr)
    file_arr.append(input_file)
    
print(filter_arr)
print(file_arr)

filter = 'F335M'

indices = [i for i, x in enumerate(filter_arr) if x == filter]
filters = np.array(filter_arr)
files = np.array(file_arr)

tmp = np.array(indices)
filters[tmp]
input_flcs = files[tmp]
input_flcs = input_flcs.tolist()
input_flcs

drz_files = glob.glob(lvl3)

drz_files

filter_arr = []
file_arr = []
for flc in drz_files:
    
    input_file= flc
    tmp = fits.open(input_file)
    fltr = tmp[0].header['FILTER']
    filter_arr.append(fltr)
    file_arr.append(input_file)
    
print(filter_arr)
print(file_arr)

indices = [i for i, x in enumerate(filter_arr) if x == filter]
filters = np.array(filter_arr)
files = np.array(file_arr)

tmp = np.array(indices)
filters[tmp]
input_drzs = files[tmp]
input_drzs = input_drzs.tolist()
input_drzs

input_flcs



In [ ]:
jwst_flc_obs = space_phot.observation2(input_flcs) 
jwst_drz_obs = space_phot.observation3(input_drzs[0])
psf_drz = space_phot.get_jwst3_psf(jwst_flc_obs, jwst_drz_obs, sn_location, num_psfs=4)
plt.imshow(psf_drz.data)
plt.show()

In [ ]:
stddevs, x_shift, y_shift, back_ground = optimize_psf(drz_obs=jwst_drz_obs, drz_psf=psf_drz, sn_loc=sn_location, fwidth=11,
                                                      xshift_range=[-2,2], yshift_range=[-2,2], background_range=[0.66,0.66],
                                                      nxs=10, nys=10, nbkg=1, plotting=False)

# x = 0.333
# y = 0.333
# background = 0.66

In [ ]:
# plot 2D array, x v. y with color as std
plt.scatter(x_shift, y_shift, c=stddevs, s=800, marker='s') # s is a size of marker 
plt.viridis()
plt.xlabel('X Shift (pixels)')
plt.ylabel('Y Shift (pixels)')
plt.colorbar(label='Standard Deviation parameter')


In [ ]:
min_ind = np.argmin(stddevs)
print('Optimized Parameters: ')
print('Standard Deviation of 2D Smoothed Background: ', stddevs[min_ind]) 
print('x_shift: ', x_shift[min_ind],'y_shift: ', y_shift[min_ind])
print('background: ',back_ground[min_ind])

plt.plot(range(len(stddevs)), stddevs, marker='o', linestyle='none')
plt.xlabel('Run Index')
plt.ylabel('stddev(2D smooth bkgd model)')

In [ ]:
stddevs, x_shift, y_shift, back_ground = optimize_psf(drz_obs=jwst_drz_obs, drz_psf=psf_drz, sn_loc=sn_location, fwidth=11,
                                                      xshift_range=[x_shift[min_ind],x_shift[min_ind]], 
                                                      yshift_range=[y_shift[min_ind],y_shift[min_ind]], 
                                                      background_range=[back_ground[min_ind],back_ground[min_ind]],
                                                      nxs=1, nys=1, nbkg=1, plotting=True)

#### Print magnitude and error

In [ ]:
print(jwst_drz_obs.psf_result.phot_cal_table)
print(jwst_drz_obs.psf_result.phot_cal_table['mag','magerr'])

mag_ab  = jwst_drz_obs.psf_result.phot_cal_table['mag'][0]
mag_err = jwst_drz_obs.psf_result.phot_cal_table['magerr'][0]

#### Calculate Flux

In [ ]:
# Upper Limit
# print('Upper Limit: ', jwst_drz_obs.upper_limit(nsigma=5))

flux_jy = 10**(-0.4 * (mag_ab + 48.6)) * 1e23
flux_err_jy = 0.4 * np.log(10) * flux_jy * mag_err
print('FLUX [Jy] = ', flux_jy)
print('FLUX ERROR [Jy] = ', flux_err_jy)


# NIRCAM F360M

In [ ]:
files = space_phot.util.filter_dict_from_list(glob.glob(lvl2),sn_location)['F360M']
files

filter_arr = []
file_arr = []
for flc in files:
    
    input_file= flc
    tmp = fits.open(input_file)
    fltr = tmp[0].header['FILTER']
    filter_arr.append(fltr)
    file_arr.append(input_file)
    
print(filter_arr)
print(file_arr)

filter = 'F360M'

indices = [i for i, x in enumerate(filter_arr) if x == filter]
filters = np.array(filter_arr)
files = np.array(file_arr)

tmp = np.array(indices)
filters[tmp]
input_flcs = files[tmp]
input_flcs = input_flcs.tolist()
input_flcs

drz_files = glob.glob(lvl3)

drz_files

filter_arr = []
file_arr = []
for flc in drz_files:
    
    input_file= flc
    tmp = fits.open(input_file)
    fltr = tmp[0].header['FILTER']
    filter_arr.append(fltr)
    file_arr.append(input_file)
    
print(filter_arr)
print(file_arr)

indices = [i for i, x in enumerate(filter_arr) if x == filter]
filters = np.array(filter_arr)
files = np.array(file_arr)

tmp = np.array(indices)
filters[tmp]
input_drzs = files[tmp]
input_drzs = input_drzs.tolist()
input_drzs

input_flcs


In [ ]:
jwst_flc_obs = space_phot.observation2(input_flcs) 
jwst_drz_obs = space_phot.observation3(input_drzs[0])
psf_drz = space_phot.get_jwst3_psf(jwst_flc_obs, jwst_drz_obs, sn_location, num_psfs=4)
plt.imshow(psf_drz.data)
plt.show()

In [ ]:
stddevs, x_shift, y_shift, back_ground = optimize_psf(drz_obs=jwst_drz_obs, drz_psf=psf_drz, sn_loc=sn_location, fwidth=11,
                                                      xshift_range=[-2,2], yshift_range=[-2,2], background_range=[0.64,0.64],
                                                      nxs=10, nys=10, nbkg=1, plotting=False)

# x = -0.66
# y = -0.4
# background = 0.64


In [ ]:
# plot 2D array, x v. y with color as std
plt.scatter(x_shift, y_shift, c=stddevs, s=800, marker='s') # s is a size of marker 
plt.viridis()
plt.xlabel('X Shift (pixels)')
plt.ylabel('Y Shift (pixels)')
plt.colorbar(label='Standard Deviation parameter')


In [ ]:
min_ind = np.argmin(stddevs)
print('Optimized Parameters: ')
print('Standard Deviation of 2D Smoothed Background: ', stddevs[min_ind]) 
print('x_shift: ', x_shift[min_ind],'y_shift: ', y_shift[min_ind])
print('background: ',back_ground[min_ind])

plt.plot(range(len(stddevs)), stddevs, marker='o', linestyle='none')
plt.xlabel('Run Index')
plt.ylabel('stddev(2D smooth bkgd model)')

In [ ]:
stddevs, x_shift, y_shift, back_ground = optimize_psf(drz_obs=jwst_drz_obs, drz_psf=psf_drz, sn_loc=sn_location, fwidth=11,
                                                      xshift_range=[x_shift[min_ind],x_shift[min_ind]], 
                                                      yshift_range=[y_shift[min_ind],y_shift[min_ind]], 
                                                      background_range=[back_ground[min_ind],back_ground[min_ind]],
                                                      nxs=1, nys=1, nbkg=1, plotting=True)

#### Print magnitude and error

In [ ]:
print(jwst_drz_obs.psf_result.phot_cal_table)
print(jwst_drz_obs.psf_result.phot_cal_table['mag','magerr'])

mag_ab  = jwst_drz_obs.psf_result.phot_cal_table['mag'][0]
mag_err = jwst_drz_obs.psf_result.phot_cal_table['magerr'][0]

#### Calculate Flux

In [ ]:
# Upper Limit
# print('Upper Limit: ', jwst_drz_obs.upper_limit(nsigma=5))

flux_jy = 10**(-0.4 * (mag_ab + 48.6)) * 1e23
flux_err_jy = 0.4 * np.log(10) * flux_jy * mag_err
print('FLUX [Jy] = ', flux_jy)
print('FLUX ERROR [Jy] = ', flux_err_jy)


# SED

In [ ]:
# Can sophie create her very first SED?!?!
x = [770, 1000, 1130, 2100, 200, 300, 335, 360]
y = [0.0000055004725286763, 0.0000387257644921614, 0.0000337287308658865, 0.000141775112312198, 7.45418213378988E-08, 1.62330452776544E-07, 3.11314966544703E-07, 3.37287308658865E-07]
yerr = [4.6390273226989E-07, 5.9210082020016E-07, 1.8909190847519416e-06, 0.0000107279857869754, 2.36175107985365E-08, 6.86259493769322E-08, 5.73463360953419E-08, 9.19533151105197E-08]

plt.errorbar(x, y, yerr=yerr, fmt='o')
plt.yscale('log')
plt.xlabel('Filter')
plt.ylabel('Flux (Jy)')
plt.title('SED of SN 1983V')
# plt.xticks(x, ['F770W', 'F1000W', 'F1130W', 'F2100W',], fontsize=8)
plt.grid()
# plt.savefig('sed_sn1983v.png', dpi=300)


In [ ]:
# axes1 = {}
# axes2 = {}


# fig, axs = plt.subplots(2,2)
# jwst_drz_obs.psf_photometry(psf_drz,sn_location,bounds={'flux':[-30000,40000],
#             'centroid':[-1,1],                              
#             'bkg':[-10,10]},
#             fit_width=11,
#             background = back_ground[min_ind],
#             fit_bkg=False,
#             fit_centroid=False, 
#             fit_flux=True,
#             centroidx_shift=x_shift[min_ind],
#             centroidy_shift=y_shift[min_ind],
#             psf_method='nest',
#                 center_weight=25,npoints=200)

# resid_cutout = jwst_drz_obs.psf_result.resid_arr[0][2:9,2:9]
# yy, xx = np.mgrid[0:resid_cutout.shape[0], 0:resid_cutout.shape[1]]
# xx = xx - resid_cutout.shape[1]//2
# yy = yy - resid_cutout.shape[0]//2

# # Fit and evaluate
# z = resid_cutout.flatten()
# x_flat = xx.flatten()
# y_flat = yy.flatten()
# m, ij = polyfit2d(x_flat, y_flat, z, deg=2)
# model = polyval2d_custom(xx, yy, m, ij)
# residuals_from_model = resid_cutout - model

# # Residual from background modeled; use this for error calculation:
# # residuals_from_model

# # But, which flux to use? 
# # the residual from the PSF fit?
# # resid_cutout
# # The model? model+bkg? data?


# # Final metric: deviation from smooth model
# smoothness_metric = np.std(residuals_from_model)
# print(f"Deviation from 2D smooth background model: {smoothness_metric:.5f}")
# ax1 = np.sum(jwst_drz_obs.psf_result.resid_arr[0][4:7],axis=0)
# ax2 = np.sum(jwst_drz_obs.psf_result.resid_arr[0].T[4:7],axis=0)

# axes1['%f,%f,%f'%(x_shift[min_ind],y_shift[min_ind],back_ground[min_ind])]=[ax1,smoothness_metric]
# axes2['%f,%f,%f'%(x_shift[min_ind],y_shift[min_ind],back_ground[min_ind])] = [ax2,smoothness_metric]

# axs[0,0].plot(np.arange(0,11,1),ax1,label='%f,%f,%f'%(x_shift[min_ind],y_shift[min_ind],back_ground[min_ind]))
# axs[0,0].legend(fontsize=8, loc='upper center')
# axs[0,1].plot(np.arange(0,11,1),ax2,label='%f,%f,%f'%(x_shift[min_ind],y_shift[min_ind],back_ground[min_ind]))
# axs[0,1].legend(fontsize=8, loc='upper center')
# for i in range(4,7):
#     ax1 = jwst_drz_obs.psf_result.resid_arr[0][i]
#     ax2 = jwst_drz_obs.psf_result.resid_arr[0].T[i]

#     axs[1,0].plot(np.arange(0,11,1),ax1,label=i)
#     axs[1,0].legend(fontsize=8)
# for i in range(4,7):
#     ax1 = jwst_drz_obs.psf_result.resid_arr[0][i]
#     ax2 = jwst_drz_obs.psf_result.resid_arr[0].T[i]
#     axs[1,1].plot(np.arange(0,11,1),ax2,label=i)
#     axs[1,1].legend(fontsize=8)
# plt.show()

# jwst_drz_obs.plot_psf_fit()
# plt.show()



# jwst_drz_obs.plot_psf_posterior()#minweight=0.001)
# plt.show()
